In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import pickle

# -------------------------------------------------
# 1. Load a pretrained CNN for image feature extraction
# -------------------------------------------------
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

def extract_features(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    features = feature_extractor.predict(x, verbose=0)
    return features  # shape: (1, 1280)

# -------------------------------------------------
# 2. Tiny vocabulary + dummy training data
#    (In a real project replace with Flickr8k / COCO captions)
# -------------------------------------------------
# Example vocabulary (word -> index)
word_to_idx = {
    '<pad>': 0, '<start>': 1, '<end>': 2,
    'a': 3, 'cat': 4, 'dog': 5, 'sitting': 6,
    'on': 7, 'the': 8, 'mat': 9, 'grass': 10
}
idx_to_word = {v: k for k, v in word_to_idx.items()}
vocab_size = len(word_to_idx)
max_length = 8          # maximum caption length

# Dummy image features + captions (replace with real data)
# features shape: (num_samples, 1280)
dummy_features = np.random.randn(4, 1280).astype('float32')
dummy_captions = [
    [1, 3, 4, 6, 7, 8, 9, 2],   # <start> a cat sitting on the mat <end>
    [1, 3, 5, 6, 7, 8, 10, 2],  # <start> a dog sitting on the grass <end>
    [1, 3, 4, 6, 7, 8, 9, 2],
    [1, 3, 5, 6, 7, 8, 10, 2]
]

def create_sequences(features, captions, max_len, vocab_size):
    X1, X2, y = [], [], []
    for i, cap in enumerate(captions):
        for t in range(1, len(cap)):
            in_seq = pad_sequences([cap[:t]], maxlen=max_len)[0]
            out_word = to_categorical([cap[t]], num_classes=vocab_size)[0]
            X1.append(features[i])
            X2.append(in_seq)
            y.append(out_word)
    return np.array(X1), np.array(X2), np.array(y)

X1, X2, y = create_sequences(dummy_features, dummy_captions, max_length, vocab_size)

# -------------------------------------------------
# 3. Build the captioning model (CNN features + LSTM)
# -------------------------------------------------
# Image feature branch
inputs1 = Input(shape=(1280,))
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# Sequence branch
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = LSTM(256)(se2)

# Decoder
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

# -------------------------------------------------
# 4. Train (very short demo – increase epochs with real data)
# -------------------------------------------------
model.fit([X1, X2], y, epochs=20, batch_size=2, verbose=1)

# -------------------------------------------------
# 5. Generate a caption for a new image
# -------------------------------------------------
def generate_caption(model, photo, max_length, word_to_idx, idx_to_word):
    in_text = '<start>'
    for _ in range(max_length):
        sequence = [word_to_idx.get(w, 0) for w in in_text.split()]
        sequence = pad_sequences([sequence], maxlen=max_length)
        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = idx_to_word.get(yhat, '')
        if word == '<end>' or word == '':
            break
        in_text += ' ' + word
    return in_text.replace('<start> ', '').replace(' <end>', '')

# Example usage (replace with a real image path)
# photo = extract_features('sample.jpg')
# caption = generate_caption(model, photo, max_length, word_to_idx, idx_to_word)
# print("Predicted caption:", caption)

print("Model trained. Use generate_caption() with real image features.")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# -------------------------------------------------
# 1. Generate synthetic sequential data with long-term dependency
#    Task: given a sequence of length T, predict the sum of the
#    first and last elements (requires remembering both ends)
# -------------------------------------------------
def generate_data(num_samples=2000, seq_len=50):
    X = np.random.randn(num_samples, seq_len, 1).astype('float32')
    # Target = first element + last element (long-range dependency)
    y = (X[:, 0, 0] + X[:, -1, 0]).reshape(-1, 1)
    return X, y

seq_length = 50
X_train, y_train = generate_data(3000, seq_length)
X_val, y_val = generate_data(500, seq_length)

# -------------------------------------------------
# 2. Design the LSTM network
# -------------------------------------------------
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(seq_length, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)                          # regression output
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# -------------------------------------------------
# 3. Train
# -------------------------------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# -------------------------------------------------
# 4. Evaluate long-term dependency learning
# -------------------------------------------------
loss, mae = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation MSE: {loss:.4f}, MAE: {mae:.4f}")

# Quick prediction demo
sample = X_val[:5]
preds = model.predict(sample, verbose=0)
print("\nSample predictions vs true values:")
for i in range(5):
    print(f"True: {y_val[i][0]:.3f}  |  Pred: {preds[i][0]:.3f}")